# QuakeTriage — Siamese U-Net Training (Kaggle GPU)

Siamese U-Net (shared ResNet34 encoder, |pre-post| skips) on KATE-CD.
Checkpoint saved to `/kaggle/working/siamese_best.pt`.

In [ ]:
# ── 1. Install dependencies ─────────────────────────────────────────────────
import subprocess, sys
def run(cmd): subprocess.check_call(cmd, shell=True)

run('pip install -q segmentation-models-pytorch albumentations huggingface_hub')

In [ ]:
# ── 2. Download KATE-CD from HuggingFace ────────────────────────────────────
from huggingface_hub import snapshot_download
import os

HF_TOKEN = os.environ.get('HF_TOKEN', '')  # set in Kaggle Secrets
snapshot_download(
    repo_id='CSCRS/kate-cd',
    repo_type='dataset',
    local_dir='/kaggle/working/kate_cd',
    token=HF_TOKEN or None,
)
print('Download complete.')

In [ ]:
# ── 3. Dataset & model (inline, mirrors src/quaketriage) ────────────────────
import io, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]
SPLITS = {
    'train':      'data/train-00000-of-00001.parquet',
    'validation': 'data/validation-00000-of-00001.parquet',
    'test':       'data/test-00000-of-00001.parquet',
}

def _decode_rgb(cell):
    raw = cell.get('bytes') if isinstance(cell, dict) else cell
    return np.array(Image.open(io.BytesIO(raw)).convert('RGB'))

def _decode_mask(cell):
    raw = cell.get('bytes') if isinstance(cell, dict) else cell
    return np.array(Image.open(io.BytesIO(raw)).convert('L'))

def _normalize(t):
    mean = torch.tensor(_MEAN).view(3,1,1)
    std  = torch.tensor(_STD ).view(3,1,1)
    return (t.float()/255.0 - mean) / std

def build_transforms(split):
    aug = [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5)] if split=='train' else []
    return A.Compose(aug + [ToTensorV2()], additional_targets={'post':'image','mask':'mask'}, is_check_shapes=False)

class KateCDDataset(Dataset):
    def __init__(self, root, split='train'):
        self.df = pd.read_parquet(f'{root}/{SPLITS[split]}')
        self.tfm = build_transforms(split)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row  = self.df.iloc[i]
        pre  = _decode_rgb(row['pre_image'])
        post = _decode_rgb(row['post_image'])
        mask = _decode_mask(row['label'])
        out  = self.tfm(image=pre, post=post, mask=mask)
        pre_t  = _normalize(out['image'])
        post_t = _normalize(out['post'])
        mask_t = (out['mask'].unsqueeze(0) > 127).float()
        return torch.cat([pre_t, post_t], 0), mask_t

class DiceFocalLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice  = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.focal = smp.losses.FocalLoss(mode='binary', alpha=0.25, gamma=2.0)
    def forward(self, logits, targets):
        return self.dice(logits, targets) + self.focal(logits, targets)

class SiameseUNet(nn.Module):
    # mirrors src/quaketriage/models/unet.py::SiameseUNet
    def __init__(self, encoder='resnet34', encoder_weights='imagenet'):
        super().__init__()
        base = smp.Unet(encoder, encoder_weights=encoder_weights, in_channels=3,
                        classes=1, activation=None)
        self.encoder = base.encoder
        self.decoder = base.decoder
        self.segmentation_head = base.segmentation_head
    def forward(self, x):
        fa = self.encoder(x[:, :3])
        fb = self.encoder(x[:, 3:])
        diff = [torch.abs(a - b) for a, b in zip(fa, fb)]
        return self.segmentation_head(self.decoder(diff))

print('Classes defined.')

In [ ]:
# ── 4. Training ──────────────────────────────────────────────────────────────
DATA_ROOT  = '/kaggle/working/kate_cd'
BATCH_SIZE = 8
MAX_EPOCHS = 50
LR         = 1e-4
PATIENCE   = 10
THRESHOLD  = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

train_ds = KateCDDataset(DATA_ROOT, 'train')
val_ds   = KateCDDataset(DATA_ROOT, 'validation')
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = SiameseUNet('resnet34', 'imagenet').to(device)
loss_fn   = DiceFocalLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_f1, patience_left = 0.0, PATIENCE
history = []

def metrics(logits, targets, thr):
    p = (torch.sigmoid(logits) > thr).float()
    tp = (p*targets).sum(); fp = (p*(1-targets)).sum(); fn = ((1-p)*targets).sum()
    pr = tp/(tp+fp+1e-8); rc = tp/(tp+fn+1e-8)
    f1 = 2*pr*rc/(pr+rc+1e-8); iou = tp/(tp+fp+fn+1e-8)
    return {'precision':pr.item(),'recall':rc.item(),'f1':f1.item(),'iou':iou.item()}

for epoch in range(1, MAX_EPOCHS+1):
    t0 = time.time()
    model.train(); tl = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(imgs), masks)
        loss.backward(); optimizer.step()
        tl += loss.item()
    tl /= len(train_loader)

    model.eval(); vl = 0; lls, ms = [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            vl += loss_fn(logits, masks).item()
            lls.append(logits.cpu()); ms.append(masks.cpu())
    vl /= len(val_loader)
    m = metrics(torch.cat(lls), torch.cat(ms), THRESHOLD)
    secs = time.time()-t0

    print(f'Epoch {epoch:3d}/{MAX_EPOCHS}  train={tl:.4f}  val={vl:.4f}  F1={m["f1"]:.4f}  IoU={m["iou"]:.4f}  ({secs:.0f}s)')
    history.append({'epoch':epoch,'train_loss':round(tl,4),'val_loss':round(vl,4),'epoch_secs':round(secs,1),**{k:round(v,4) for k,v in m.items()}})

    if m['f1'] > best_f1:
        best_f1 = m['f1']; patience_left = PATIENCE
        torch.save({'epoch':epoch,'state_dict':model.state_dict(),'best_f1':best_f1,'threshold':THRESHOLD},
                   '/kaggle/working/siamese_best.pt')
        print(f'  ✓ New best F1={best_f1:.4f}')
    else:
        patience_left -= 1
        if patience_left == 0:
            print(f'Early stopping at epoch {epoch}.'); break

print(f'\nDone. Best val F1={best_f1:.4f}')

In [ ]:
# ── 5. Save history + threshold tuning ───────────────────────────────────────
import json

# Threshold sweep on validation set
ckpt = torch.load('/kaggle/working/siamese_best.pt', map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()
all_logits, all_masks = [], []
with torch.no_grad():
    for imgs, masks in val_loader:
        logits = model(imgs.to(device))
        all_logits.append(logits.cpu()); all_masks.append(masks.cpu())
all_logits = torch.cat(all_logits); all_masks = torch.cat(all_masks)

best_thr, best_thr_f1 = 0.5, 0.0
for thr in [i/20 for i in range(1, 20)]:
    m = metrics(all_logits, all_masks, thr)
    print(f'  thr={thr:.2f}  F1={m["f1"]:.4f}  prec={m["precision"]:.4f}  rec={m["recall"]:.4f}')
    if m['f1'] > best_thr_f1:
        best_thr_f1 = m['f1']; best_thr = thr
print(f'\nBest threshold: {best_thr}  (val F1={best_thr_f1:.4f})')

with open('/kaggle/working/training_history_siamese.json', 'w') as f:
    json.dump({'history': history, 'best_threshold': best_thr, 'best_val_f1': best_thr_f1}, f, indent=2)
print('history saved.')